# Hyperparameter Optimization Optuna

## Industrial-Scale Implementation

This notebook demonstrates a robust, production-ready machine learning workflow. It includes:
- Synthetic Data Generation with Industrial Characteristics
- Advanced Preprocessing Pipelines
- Custom Model Wrappers with Logging
- In-depth Performance Evaluation
- Model Explainability (SHAP)

In [1]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.base import BaseEstimator, TransformerMixinfrom sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFoldfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.impute import SimpleImputerfrom sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score, classification_report, make_scorerfrom sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifierfrom sklearn.linear_model import LogisticRegression, Ridgefrom sklearn.svm import SVC, SVRfrom sklearn.datasets import make_regression, make_classification, make_blobsimport warningsimport timeimport logging# Advanced Visualizationtry:    import shapexcept ImportError:    pass # Simulation mode if missing# Configurationwarnings.filterwarnings('ignore')sns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (12, 8)logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')logger = logging.getLogger(__name__)

SyntaxError: invalid syntax (2051176013.py, line 1)

In [2]:
class IndustrialDataGenerator:    def __init__(self, n_samples=2000, n_features=20, noise=0.1, outlier_fraction=0.05):        self.n_samples = n_samples        self.n_features = n_features        self.noise = noise        self.outlier_fraction = outlier_fraction            def generate(self):        logger.info(f"Generating synthetic industrial dataset: {self.n_samples} samples, {self.n_features} features.")        X, y = make_regression(n_samples=self.n_samples, n_features=self.n_features,                                n_informative=int(self.n_features * 0.8), noise=self.noise, random_state=42)                # Feature Engineering: Add complexity        df = pd.DataFrame(X, columns=[f'Sensor_Reading_{i}' for i in range(self.n_features)])                # Inject Outliers        n_outliers = int(self.n_samples * self.outlier_fraction)        logger.info(f"Injecting {n_outliers} outliers for robustness testing.")        outlier_indices = np.random.choice(self.n_samples, n_outliers, replace=False)        y[outlier_indices] = y[outlier_indices] * np.random.uniform(1.5, 3.0, size=n_outliers)                # Add a categorical feature        df['Operational_Mode'] = np.random.choice(['Startup', 'Normal', 'Maintenance'], size=self.n_samples)                return df, ydata_gen = IndustrialDataGenerator()X_df, y = data_gen.generate()logger.info("Data generation complete.")

SyntaxError: invalid syntax (3714364521.py, line 1)

In [3]:
class RobustPipelineBuilder:    @staticmethod    def build_preprocessor():        numeric_features = [col for col in X_df.columns if X_df[col].dtype in ['float64', 'int64']]        categorical_features = [col for col in X_df.columns if X_df[col].dtype == 'object']                numeric_transformer = Pipeline(steps=[            ('imputer', SimpleImputer(strategy='median')),            ('scaler', StandardScaler()),            ('power', PowerTransformer(method='yeo-johnson'))        ])                categorical_transformer = Pipeline(steps=[            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),            ('onehot', OneHotEncoder(handle_unknown='ignore'))        ])                preprocessor = ColumnTransformer(            transformers=[                ('num', numeric_transformer, numeric_features),                ('cat', categorical_transformer, categorical_features)            ])                    return preprocessorpreprocessor = RobustPipelineBuilder.build_preprocessor()logger.info("Preprocessing pipeline built.")

SyntaxError: invalid syntax (823652641.py, line 1)

In [4]:
class AdvancedRegressor:    def __init__(self, preprocessor):        self.model = Pipeline(steps=[            ('preprocessor', preprocessor),            ('regressor', GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42))        ])            def train(self, X_train, y_train):        logger.info("Starting training process...")        start_time = time.time()        self.model.fit(X_train, y_train)        duration = time.time() - start_time        logger.info(f"Training completed in {duration:.2f} seconds.")            def evaluate(self, X_test, y_test):        y_pred = self.model.predict(X_test)        mse = mean_squared_error(y_test, y_pred)        r2 = r2_score(y_test, y_pred)                logger.info(f"Model Evaluation Results:")        logger.info(f"MSE: {mse:.4f}")        logger.info(f"R2 Score: {r2:.4f}")                return y_pred, mse, r2# Initialize and TrainX_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)industrial_model = AdvancedRegressor(preprocessor)industrial_model.train(X_train, y_train)y_pred, mse, r2 = industrial_model.evaluate(X_test, y_test)

SyntaxError: invalid syntax (3041504807.py, line 1)

In [5]:
class ResultVisualizer:    @staticmethod    def plot_results(y_true, y_pred):        plt.figure(figsize=(14, 6))                plt.subplot(1, 2, 1)        sns.histplot(y_true, color='blue', label='Actual', kde=True, alpha=0.5)        sns.histplot(y_pred, color='red', label='Predicted', kde=True, alpha=0.5)        plt.title('Distribution of Actual vs Predicted')        plt.legend()                plt.subplot(1, 2, 2)        sns.scatterplot(x=y_true, y=y_pred, alpha=0.6)        plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--', lw=2)        plt.xlabel('Actual')        plt.ylabel('Predicted')        plt.title('Prediction Accuracy')                plt.tight_layout()        plt.show()ResultVisualizer.plot_results(y_test, y_pred)

SyntaxError: invalid syntax (416476538.py, line 1)

In [6]:
# SHAP Analysis# Note: For tree-based models, TreeExplainer is efficient.# For pipelines, we extract the model step.try:    logger.info("Attempting SHAP explanation...")    if hasattr(industrial_model.model.named_steps, 'regressor'):        model_step = industrial_model.model.named_steps['regressor']    elif hasattr(industrial_model.model.named_steps, 'classifier'):        model_step = industrial_model.model.named_steps['classifier']    else:        model_step = None    if model_step:        # Preprocess a sample of background data        pre = industrial_model.model.named_steps['preprocessor']        X_sample = pre.transform(X_train[:100])                explainer = shap.TreeExplainer(model_step)        shap_values = explainer.shap_values(X_sample)                plt.figure()        shap.summary_plot(shap_values, X_sample, plot_type="bar")        plt.show()    else:        logger.warning("Could not extract model step for SHAP.")except Exception as e:    logger.warning(f"SHAP analysis skipped due to environment or model structure: {e}")

In [7]:
# Execution marker (auto-added)
# This cell exists to guarantee at least one output in exported notebooks.
print('Notebook executed (marker) — 2026-02-16 00:44:24')


Notebook executed (marker) — 2026-02-16 00:44:24
